In [1]:
import numpy as np
import scipy.sparse as sp
from scipy.optimize import linprog
from scipy.sparse.linalg import spsolve
import time

# CVXPY is optional - only needed if you uncomment the CVXPY solve section
try:
    import cvxpy as cp
    HAS_CVXPY = True
except ImportError:
    HAS_CVXPY = False
    print("Note: CVXPY not installed. Install with 'uv pip install cvxpy' if needed.")
    print()

def create_random_lp(n_vars, n_constraints, density=0.1, seed=42):
    """
    Create a random linear program with non-trivial solution:
    minimize c^T x
    subject to A x <= b
               lb <= x <= ub
    
    Returns problem in standard form for interior point method
    """
    np.random.seed(seed)
    
    # Objective function coefficients (mix of positive and negative)
    c = np.random.randn(n_vars)
    
    # Constraint matrix (sparse)
    A_ub = sp.random(n_constraints, n_vars, density=density, format='csr')
    A_ub.data = np.random.randn(A_ub.nnz)
    
    # Create a feasible interior point first
    x_interior = np.random.rand(n_vars) * 8 + 1  # Between 1 and 9
    
    # Right-hand side (ensure x_interior is feasible with some slack)
    slack = np.random.rand(n_constraints) * 3 + 2
    b_ub = A_ub @ x_interior + slack
    
    # Bounds on variables
    bounds = [(0, 10) for _ in range(n_vars)]
    
    return c, A_ub, b_ub, bounds

def solve_lp_scipy(c, A, b, bounds):
    """
    Solve LP using scipy's interior point method
    """
    print("Solving with scipy.optimize.linprog (interior-point)...")
    
    start = time.time()
    
    # scipy expects: minimize c^T x subject to A_ub x <= b_ub
    result = linprog(c, A_ub=A, b_ub=b, bounds=bounds, method='highs-ipm', 
                     options={'disp': False})  # Set to False for cleaner output
    
    elapsed = time.time() - start
    
    return result, elapsed

def solve_lp_cvxpy(c, A, b, bounds):
    """
    Solve LP using CVXPY (uses interior point methods like SCS or ECOS)
    """
    if not HAS_CVXPY:
        return None, None, None
    
    print("\nSolving with CVXPY (ECOS interior-point solver)...")
    
    n_vars = len(c)
    
    # Define optimization variable
    x = cp.Variable(n_vars)
    
    # Convert sparse matrix to dense for CVXPY (or use @ operator)
    A_dense = A.toarray()
    
    # Define objective and constraints
    objective = cp.Minimize(c @ x)
    constraints = [A_dense @ x <= b]
    
    # Add bounds
    for i in range(n_vars):
        constraints.append(x[i] >= bounds[i][0])
        constraints.append(x[i] <= bounds[i][1])
    
    # Create and solve problem
    prob = cp.Problem(objective, constraints)
    
    start = time.time()
    # Try multiple solvers in order of preference
    try:
        prob.solve(solver=cp.CLARABEL, verbose=False)
    except:
        try:
            prob.solve(solver=cp.SCS, verbose=False)
        except:
            prob.solve(verbose=False)  # Use default solver
    elapsed = time.time() - start
    
    return x.value, prob.value, elapsed

def analyze_kkt_system(c, A, b, x, lam, s):
    """
    Analyze the KKT system that interior point methods solve.
    At each iteration, IPM solves a system of the form:
    
    [ H    A^T ] [ dx   ]   [ r_dual ]
    [ A   -D^-1] [ dlam ] = [ r_prim ]
    
    where H is related to the Hessian (for LP, H=0)
    D is a diagonal matrix related to slack variables
    
    For LP, this reduces to solving: A D A^T dlam = rhs
    where D is diagonal, related to x and slack variables
    """
    m, n = A.shape
    
    # Simulated barrier parameter (decreases each iteration)
    mu = 0.1
    
    # Diagonal scaling matrix (simplified - in real IPM this is x./s)
    # where s are slack variables
    D = sp.diags(np.random.rand(n) + 0.1)
    
    # Form the normal equations matrix: A D A^T
    # This is what gets factorized at each IPM iteration
    ADA = A @ D @ A.T
    
    print("\n" + "="*70)
    print("KKT System Analysis")
    print("="*70)
    print(f"Constraint matrix A: {m} × {n}")
    print(f"  Non-zeros: {A.nnz:,}")
    print(f"  Density: {A.nnz/(m*n)*100:.3f}%")
    print()
    print(f"Normal equations matrix (A D A^T): {m} × {m}")
    print(f"  Non-zeros: {ADA.nnz:,}")
    print(f"  Density: {ADA.nnz/(m*m)*100:.3f}%")
    print(f"  Fill-in ratio: {ADA.nnz / A.nnz:.2f}×")
    print()
    
    # Time solving the normal equations (what happens each IPM iteration)
    rhs = np.random.randn(m)
    
    print("Solving A D A^T dlam = rhs (one IPM iteration)...")
    start = time.time()
    dlam = spsolve(ADA, rhs)
    elapsed = time.time() - start
    print(f"  Time: {elapsed:.4f} seconds")
    print(f"  This would happen ~10-50 times in a full IPM solve")
    print("="*70)
    
    return ADA

# Main example
if __name__ == "__main__":
    print("Interior Point Method for Linear Programming")
    print("="*70)
    print()
    
    # Create a moderately sized LP problem
    # Try different sizes to see scaling
    problems = [
        ("Small", 100, 50, 0.2),
        ("Medium", 500, 200, 0.1),
        ("Large", 2000, 800, 0.05),
    ]
    
    for name, n_vars, n_constraints, density in problems:
        print(f"\n{name} Problem: {n_vars} variables, {n_constraints} constraints")
        print("-"*70)
        
        c, A, b, bounds = create_random_lp(n_vars, n_constraints, density)
        
        # Solve with scipy
        result_scipy, time_scipy = solve_lp_scipy(c, A, b, bounds)
        
        if result_scipy.success:
            print(f"\nScipy solution:")
            print(f"  Status: {result_scipy.message}")
            print(f"  Optimal value: {result_scipy.fun:.6f}")
            print(f"  Solve time: {time_scipy:.4f} seconds")
            print(f"  Iterations: {result_scipy.nit}")
            
            # Analyze the KKT system structure
            # Use dummy values for x, lambda, s
            x_dummy = np.maximum(result_scipy.x, 0.01)
            lam_dummy = np.random.rand(n_constraints) + 0.1
            s_dummy = np.random.rand(n_vars) + 0.1
            
            ADA = analyze_kkt_system(c, A, b, x_dummy, lam_dummy, s_dummy)
        else:
            print(f"\nScipy failed: {result_scipy.message}")
        
        # Optional: solve with CVXPY
        if HAS_CVXPY and name == "Small":  # Only for small problem to save time
            x_cvx, obj_cvx, time_cvx = solve_lp_cvxpy(c, A, b, bounds)
            if x_cvx is not None:
                print(f"\nCVXPY solution:")
                print(f"  Optimal value: {obj_cvx:.6f}")
                print(f"  Solve time: {time_cvx:.4f} seconds")
        
        print("\n" + "="*70)
    
    print("\n\nKey Observations:")
    print("-"*70)
    print("1. The constraint matrix A is very sparse (< 10% density)")
    print("2. The normal equations A D A^T has MORE fill-in")
    print("3. Each IPM iteration solves a linear system with A D A^T")
    print("4. This system is solved via Cholesky factorization")
    print("5. The system becomes ill-conditioned as IPM approaches optimality")
    print("6. This is why direct methods (Cholesky) are preferred over CG")

Note: CVXPY not installed. Install with 'uv pip install cvxpy' if needed.

Interior Point Method for Linear Programming


Small Problem: 100 variables, 50 constraints
----------------------------------------------------------------------
Solving with scipy.optimize.linprog (interior-point)...

Scipy solution:
  Status: Optimization terminated successfully. (HiGHS Status 7: Optimal)
  Optimal value: -382.232077
  Solve time: 0.0023 seconds
  Iterations: 12

KKT System Analysis
Constraint matrix A: 50 × 100
  Non-zeros: 1,000
  Density: 20.000%

Normal equations matrix (A D A^T): 50 × 50
  Non-zeros: 2,458
  Density: 98.320%
  Fill-in ratio: 2.46×

Solving A D A^T dlam = rhs (one IPM iteration)...
  Time: 0.0011 seconds
  This would happen ~10-50 times in a full IPM solve


Medium Problem: 500 variables, 200 constraints
----------------------------------------------------------------------
Solving with scipy.optimize.linprog (interior-point)...

Scipy solution:
  Status: Optimization ter